In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("pandas:", pd.__version__)

pandas: 3.0.6


In [2]:
import pandas as pd
from pathlib import Path

data_dir = Path("Documents/CS5488 bigdata/airquality_hk/data")  # 确认这是你的数据文件夹路径
all_df = []

for f in data_dir.glob("*.csv"):
    try:
        # skiprows=7: 跳过前7行说明，第8行作为表头 (header=0 是默认值)
        # on_bad_lines="skip": 兜底跳过极个别格式错乱行，防止中断
        df = pd.read_csv(f, skiprows=7, on_bad_lines="skip")
        
        # 标记来源文件，方便后续排查数据归属
        df["_source_file"] = f.name
        print(f"✅ 读取成功: {f.name} | 形状: {df.shape} | 列数: {len(df.columns)}")
        all_df.append(df)
        
    except Exception as e:
        print(f"❌ 读取失败: {f.name} | 错误: {e}")

# 合并所有文件
if all_df:
    raw = pd.concat(all_df, ignore_index=True)
    print("\n📊 合并完成！")
    print("总形状:", raw.shape)
    print("\n列名列表（请核对）:")
    print(raw.columns.tolist())
    print("\n前几行预览:")
    display(raw.head())  # JupyterLab 中用 display 展示更美观
else:
    print("⚠️ 未读取到任何 CSV 数据，请检查 ./aqhi_raw 路径及文件后缀")

✅ 读取成功: 202003_Eng.csv | 形状: (775, 19) | 列数: 19
✅ 读取成功: 202307_Eng.csv | 形状: (775, 21) | 列数: 21
✅ 读取成功: 202109_Eng.csv | 形状: (750, 21) | 列数: 21
✅ 读取成功: 201910_Eng.csv | 形状: (775, 19) | 列数: 19
✅ 读取成功: 201503_Eng.csv | 形状: (775, 18) | 列数: 18
✅ 读取成功: 201409_Eng.csv | 形状: (750, 18) | 列数: 18
✅ 读取成功: 201607_Eng.csv | 形状: (775, 19) | 列数: 19
✅ 读取成功: 202405_Eng.csv | 形状: (775, 21) | 列数: 21
✅ 读取成功: 201312_Eng.csv | 形状: (50, 18) | 列数: 18
✅ 读取成功: 202110_Eng.csv | 形状: (775, 21) | 列数: 21
✅ 读取成功: 202204_Eng.csv | 形状: (750, 21) | 列数: 21
✅ 读取成功: 201410_Eng.csv | 形状: (775, 18) | 列数: 18
✅ 读取成功: 202602_Eng.csv | 形状: (700, 21) | 列数: 21
✅ 读取成功: 201803_Eng.csv | 形状: (775, 19) | 列数: 19
✅ 读取成功: 201909_Eng.csv | 形状: (750, 19) | 列数: 19
✅ 读取成功: 201704_Eng.csv | 形状: (750, 19) | 列数: 19
✅ 读取成功: 202506_Eng.csv | 形状: (750, 21) | 列数: 21
✅ 读取成功: 202111_Eng.csv | 形状: (750, 21) | 列数: 21
✅ 读取成功: 202101_Eng.csv | 形状: (775, 21) | 列数: 21
✅ 读取成功: 202205_Eng.csv | 形状: (775, 21) | 列数: 21
✅ 读取成功: 201802_Eng.csv | 形状: (700, 19) | 

,Date,Hour,Central/Western,Eastern,Kwun Tong,Sham Shui Po,Kwai Chung,Tsuen Wan,Tseung Kwan O,Yuen Long,...,Tung Chung,Tai Po,Sha Tin,Tap Mun,Causeway Bay,Central,Mong Kok,_source_file,Southern,North
0,2020-03-01,01,3,3,3,3,3,2,2,3,...,2,2,2,2,3,4,3,202003_Eng.csv,NaN,NaN
1,NaN,02,3,3,3,3,3,2,2,3,...,2,2,2,2,3,4,3,202003_Eng.csv,NaN,NaN
2,NaN,03,3,3,3,3,2,2,2,3,...,2,2,2,2,3,4,3,202003_Eng.csv,NaN,NaN
3,NaN,04,3,3,3,3,2,2,2,2,...,2,2,2,2,3,3,3,202003_Eng.csv,NaN,NaN
4,NaN,05,3,3,3,3,2,2,2,2,...,2,2,2,2,3,3,3,202003_Eng.csv,NaN,NaN


In [3]:
# 1. 日期向前填充
raw["Date"] = raw["Date"].ffill()

# 2. 去掉 Daily Max 行（不是小时数据）
before = len(raw)
raw = raw[raw["Hour"] != "Daily Max"].copy()
print(f"去掉 Daily Max 行: {before} → {len(raw)} (删了 {before - len(raw)} 行)")

# 3. Hour 转数值并减 1（1~24 → 0~23）
raw["Hour"] = raw["Hour"].astype(int) - 1  # 现在 Hour 是 0~23

# 4. 拼接 datetime
raw["datetime"] = pd.to_datetime(
    raw["Date"].astype(str) + " " + raw["Hour"].astype(str).str.zfill(2) + ":00",
    errors="coerce"
)

# 5. 检查
print(f"\n时间范围: {raw['datetime'].min()} ~ {raw['datetime'].max()}")
print(f"解析失败行数: {raw['datetime'].isna().sum()}")
print(f"\n前10行:")
raw[["Date", "Hour", "datetime"]].head(10)

去掉 Daily Max 行: 113975 → 109416 (删了 4559 行)

时间范围: 2013-12-30 00:00:00 ~ 2026-06-30 23:00:00
解析失败行数: 0

前10行:


,Date,Hour,datetime
0,2020-03-01,0,2020-03-01 00:00:00
1,2020-03-01,1,2020-03-01 01:00:00
2,2020-03-01,2,2020-03-01 02:00:00
3,2020-03-01,3,2020-03-01 03:00:00
4,2020-03-01,4,2020-03-01 04:00:00
5,2020-03-01,5,2020-03-01 05:00:00
6,2020-03-01,6,2020-03-01 06:00:00
7,2020-03-01,7,2020-03-01 07:00:00
8,2020-03-01,8,2020-03-01 08:00:00
9,2020-03-01,9,2020-03-01 09:00:00


In [4]:
# 站名列 = 所有不是 Date/Hour/datetime/_source_file 的列
id_cols = ["datetime", "_source_file"]
station_cols = [c for c in raw.columns if c not in ["Date", "Hour", "datetime", "_source_file"]]

long = raw.melt(
    id_vars=["datetime", "_source_file"],
    value_vars=station_cols,
    var_name="station",
    value_name="aqhi_raw"
)

print(f"长表形状: {long.shape}")
print(f"\n站点列表 ({len(long['station'].unique())} 个):")
print(sorted(long['station'].unique()))
print(f"\n前10行:")
long.head(10)

长表形状: (1969488, 4)

站点列表 (18 个):
['Causeway Bay', 'Central', 'Central/Western', 'Eastern', 'Kwai Chung', 'Kwun Tong', 'Mong Kok', 'North', 'Sha Tin', 'Sham Shui Po', 'Southern', 'Tai Po', 'Tap Mun', 'Tseung Kwan O', 'Tsuen Wan', 'Tuen Mun', 'Tung Chung', 'Yuen Long']

前10行:


,datetime,_source_file,station,aqhi_raw
0,2020-03-01 00:00:00,202003_Eng.csv,Central/Western,3
1,2020-03-01 01:00:00,202003_Eng.csv,Central/Western,3
2,2020-03-01 02:00:00,202003_Eng.csv,Central/Western,3
3,2020-03-01 03:00:00,202003_Eng.csv,Central/Western,3
4,2020-03-01 04:00:00,202003_Eng.csv,Central/Western,3
5,2020-03-01 05:00:00,202003_Eng.csv,Central/Western,3
6,2020-03-01 06:00:00,202003_Eng.csv,Central/Western,3
7,2020-03-01 07:00:00,202003_Eng.csv,Central/Western,3
8,2020-03-01 08:00:00,202003_Eng.csv,Central/Western,3
9,2020-03-01 09:00:00,202003_Eng.csv,Central/Western,3


In [5]:
# 在 Cell 5 的 parse_aqhi 里已经能区分，加一列即可
def parse_aqhi_v2(val):
    if pd.isna(val):
        return np.nan, False, False
    s = str(val).strip()
    is_plus = False
    imputed = False
    if s.endswith("*"):
        imputed = True
        s = s.rstrip("*")
    if s.endswith("+"):
        is_plus = True
        s = s.rstrip("+")
    if s in ("", "-", "N/A", "NA", "nan"):
        return np.nan, False, False
    try:
        return float(s), imputed, is_plus
    except ValueError:
        return np.nan, False, False

parsed = [parse_aqhi_v2(v) for v in long["aqhi_raw"]]
long["aqhi"] = [x[0] for x in parsed]
long["imputed"] = [x[1] for x in parsed]
long["is_10plus"] = [x[2] for x in parsed]  # 新增：标记真正的 10+

print(f"10+ 标记数量: {long['is_10plus'].sum():,}")
print(f"占 AQHI≥10 的比例: {long['is_10plus'].sum() / (long['aqhi']==10).sum()*100:.1f}%")

10+ 标记数量: 1,772
占 AQHI≥10 的比例: 46.9%


In [6]:
# 去重（同一站+同一时间）
long = long.drop_duplicates(subset=["datetime", "station"])

# 裁剪到 2013-12 ~ 2026-06
lo = pd.Timestamp("2013-12-01 00:00:00")
hi = pd.Timestamp("2026-06-30 23:00:00")
long = long[(long["datetime"] >= lo) & (long["datetime"] <= hi)]

# 排序
long = long.sort_values(["station", "datetime"]).reset_index(drop=True)

print(f"最终形状: {long.shape}")
print(f"时间范围: {long['datetime'].min()} ~ {long['datetime'].max()}")
print(f"站点数: {long['station'].nunique()}")
print(f"\n各站数据量:")
long.groupby("station").size().sort_values(ascending=False)

最终形状: (1969488, 7)
时间范围: 2013-12-30 00:00:00 ~ 2026-06-30 23:00:00
站点数: 18

各站数据量:


station
Causeway Bay       109416
Central            109416
Tung Chung         109416
Tuen Mun           109416
Tsuen Wan          109416
Tseung Kwan O      109416
Tap Mun            109416
Tai Po             109416
Southern           109416
Sham Shui Po       109416
Sha Tin            109416
North              109416
Mong Kok           109416
Kwun Tong          109416
Kwai Chung         109416
Eastern            109416
Central/Western    109416
Yuen Long          109416
dtype: int64

In [7]:
# 最终列选择（保留 is_10plus，去掉原始 aqhi_raw）
final = long[["datetime", "station", "aqhi", "imputed", "is_10plus"]].copy()

# 长表 - 主数据（带 is_10plus）
final.to_parquet("aqhi_clean_long.parquet", index=False)
final.to_csv("aqhi_clean_long.csv", index=False)

# 宽表 - 站为列（aqhi 透视；is_10plus 一般不在宽表做透视，避免含义失真）
wide = final.dropna(subset=["aqhi"]).pivot_table(
    index="datetime",
    columns="station",
    values="aqhi",
    aggfunc="first"
)
wide.to_parquet("aqhi_clean_wide.parquet", index=False)
wide.to_csv("aqhi_clean_wide.csv")

# 日聚合 - 月均/年均值分析用（保留 is_10plus 标记：是否含10+、10+小时数等）
daily = final.dropna(subset=["aqhi"]).groupby(
    ["station", final["datetime"].dt.date]
).agg(
    aqhi_daily_mean=("aqhi", "mean"),
    is_10plus_any=("is_10plus", "any"),      # 当天是否出现 10+
    is_10plus_hours=("is_10plus", "sum")     # 当天 10+ 小时数（可选）
).reset_index()
daily = daily.rename(columns={"datetime": "date"})  # groupby 后列名已对应
daily.to_parquet("aqhi_clean_daily.parquet", index=False)

print("✅ 输出完成（含 is_10plus）！")
print(f"   长表: aqhi_clean_long.parquet ({final.shape[0]:,} 行 × {final.shape[1]} 列)")
print(f"   宽表: aqhi_clean_wide.parquet ({wide.shape[0]:,} 行 × {wide.shape[1]} 列)")
print(f"   日聚合: aqhi_clean_daily.parquet ({daily.shape[0]:,} 行)")
print(f"   长表中 is_10plus 数量: {final['is_10plus'].sum():,}")

✅ 输出完成（含 is_10plus）！
   长表: aqhi_clean_long.parquet (1,969,488 行 × 5 列)
   宽表: aqhi_clean_wide.parquet (108,898 行 × 18 列)
   日聚合: aqhi_clean_daily.parquet (76,045 行)
   长表中 is_10plus 数量: 1,772


In [8]:
# =====================================================================
# Cell 8: Feature Creation
# 读取清洗好的长表，生成时间特征、站点属性、衍生标签
# =====================================================================

import pandas as pd
import numpy as np

# 1. 读取清洗好的数据
df = pd.read_parquet("aqhi_clean_long.parquet")
print(f"读取: {df.shape[0]:,} 行 × {df.shape[1]} 列")
print(f"时间范围: {df['datetime'].min()} ~ {df['datetime'].max()}")
print(f"站点数: {df['station'].nunique()}")

# 2. 时间基础特征 -------------------------------------------------------
df["hour"] = df["datetime"].dt.hour          # 0~23 小时，用于小时分布图
df["dow"]  = df["datetime"].dt.dayofweek     # 0=周一 ... 6=周日

# 精细日历：区分 普通工作日 / 周六 / 周日 / 公众假日
# （香港公众假日表，可自行补充年份；此处给 2020-2026 示例）
# 读取官方公众假期表
hol = pd.read_csv("Documents/CS5488 bigdata/airquality_hk/cleaning/hk_public_holidays_2013_2026.csv", parse_dates=["date"])
holiday_set = set(hol["date"].dt.date)

df["is_public_holiday"] = df["datetime"].dt.date.isin(holiday_set).astype(int)

def calendar_type(row):
    if row["is_public_holiday"]:
        return "PublicHoliday"
    if row["dow"] == 6:
        return "Sunday"
    if row["dow"] == 5:
        return "Saturday"
    return "Weekday"

df["day_type"] = df.apply(calendar_type, axis=1)

print(df["day_type"].value_counts())

# 季节（气象季节）
season_map = {1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
              6:"Summer",7:"Summer",8:"Summer",9:"Autumn",10:"Autumn",
              11:"Autumn",12:"Winter"}
df["season"] = df["datetime"].dt.month.map(season_map)

# 交通高峰时段：7:00~9:00 与 17:00~19:00
df["is_rush_hour"] = df["hour"].isin([7,8,9,17,18,19]).astype(int)

# 3. 站点属性（采用环保署官方口径：路边站 vs 一般站） ----------------------
roadside_stations = {"Central", "Mong Kok", "Causeway Bay"}
df["station_type"] = df["station"].isin(roadside_stations).map(
    {True: "roadside", False: "general"})
df["is_roadside"] = (df["station_type"] == "roadside").astype(int)

# station_id：标签编码（模型输入用）
station_label = {s: i for i, s in enumerate(df["station"].unique())}
df["station_id"] = df["station"].map(station_label)

# 4. 衍生标签 -----------------------------------------------------------
df["is_high"] = (df["aqhi"] >= 7).astype(int)   # 健康风险高

# 连续超标小时数（按站分组计算）
df = df.sort_values(["station", "datetime"]).reset_index(drop=True)
df["_is_clean"] = (df["is_high"] == 0).astype(int)
df["_group"] = (df["_is_clean"] != df["_is_clean"].shift(1)).cumsum()
df["high_hours_count"] = df.groupby(["station", "_group"]).cumcount() + 1
df.loc[df["is_high"] == 0, "high_hours_count"] = 0

# 5. 清理中间列，查看结果 -----------------------------------------------
df = df.drop(columns=["_is_clean", "_group"])
print(f"\n特征工程完成，最终形状: {df.shape}")
print("\n新增特征列:")
new_cols = ["hour","dow","is_public_holiday","day_type","season",
            "is_rush_hour","station_type","is_roadside","zone_type",
            "station_id","is_high","high_hours_count"]
for c in new_cols:
    print(f"  - {c}")

print("\nday_type 分布:")
print(df["day_type"].value_counts())
print("\nstation_type 分布:")
print(df.groupby("station_type")["station"].nunique())

# 6. 输出
df.to_parquet("aqhi_features.parquet", index=False)
print(f"\n✅ 已保存 aqhi_features.parquet ({df.shape[0]:,} 行 × {df.shape[1]} 列)")

读取: 1,969,488 行 × 5 列
时间范围: 2013-12-30 00:00:00 ~ 2026-06-30 23:00:00
站点数: 18
day_type
Weekday          1329696
Sunday            281232
Saturday          265248
PublicHoliday      93312
Name: count, dtype: int64

特征工程完成，最终形状: (1969488, 16)

新增特征列:
  - hour
  - dow
  - is_public_holiday
  - day_type
  - season
  - is_rush_hour
  - station_type
  - is_roadside
  - zone_type
  - station_id
  - is_high
  - high_hours_count

day_type 分布:
day_type
Weekday          1329696
Sunday            281232
Saturday          265248
PublicHoliday      93312
Name: count, dtype: int64

station_type 分布:
station_type
general     15
roadside     3
Name: station, dtype: int64

✅ 已保存 aqhi_features.parquet (1,969,488 行 × 16 列)
